# Load Automation Logs With pandas

This notebook loads `logs/automation.log` into a pandas DataFrame and creates a few quick summaries.

In [ ]:
from pathlib import Path

import pandas as pd

pd.set_option("display.max_colwidth", 160)

In [ ]:
log_path = Path("../logs/automation.log")

if not log_path.exists():
    log_path = Path("logs/automation.log")

log_path.resolve()

In [ ]:
logs = pd.read_csv(
    log_path,
    sep=r"\s*\|\s*",
    engine="python",
    names=["time", "level", "logger", "process", "message"],
    dtype="string",
)

logs.head()

In [ ]:
logs.info()

In [ ]:
logs["level"].value_counts(dropna=False)

In [ ]:
errors = logs[logs["level"].isin(["ERROR", "CRITICAL", "WARNING"])]
errors.head(20)

In [ ]:
logs.groupby(["level", "logger"], dropna=False).size().sort_values(ascending=False).head(20)

## Filter and count events by IP address

Extract IPv4 addresses from the log message, keep only events that contain an IP, and count events per IP address.

In [ ]:
ip_pattern = r"\b(?:25[0-5]|2[0-4]\d|1?\d?\d)(?:\.(?:25[0-5]|2[0-4]\d|1?\d?\d)){3}\b"

logs_with_ip = logs.assign(
    ip=logs["message"].str.extract(f"({ip_pattern})", expand=False)
).dropna(subset=["ip"])

logs_with_ip.head(20)

In [ ]:
events_by_ip = (
    logs_with_ip.groupby("ip", dropna=False)
    .size()
    .reset_index(name="event_count")
    .sort_values("event_count", ascending=False)
)

if events_by_ip.empty:
    print("No IPv4 addresses found in this log file.")
else:
    display(events_by_ip)

In [ ]:
# Optional: count by IP and log level
events_by_ip_and_level = (
    logs_with_ip.groupby(["ip", "level"], dropna=False)
    .size()
    .reset_index(name="event_count")
    .sort_values(["event_count", "ip"], ascending=[False, True])
)

events_by_ip_and_level

## Suspicious activity by hour

The default suspicious filter counts events with warning/error levels or messages that contain suspicious keywords. Adjust `suspicious_keywords` for your log source.

In [ ]:
import matplotlib.pyplot as plt

suspicious_levels = {"WARNING", "ERROR", "CRITICAL"}
suspicious_keywords = [
    "failed",
    "failure",
    "error",
    "denied",
    "unauthorized",
    "attack",
    "suspicious",
    "malware",
    "blocked",
]

In [ ]:
logs_by_hour = logs.copy()
logs_by_hour["hour"] = pd.to_datetime(logs_by_hour["time"], format="%H:%M:%S", errors="coerce").dt.hour

keyword_pattern = "|".join(suspicious_keywords)
logs_by_hour["is_suspicious"] = (
    logs_by_hour["level"].isin(suspicious_levels)
    | logs_by_hour["message"].str.contains(keyword_pattern, case=False, na=False, regex=True)
)

suspicious_by_hour = (
    logs_by_hour[logs_by_hour["is_suspicious"]]
    .groupby("hour")
    .size()
    .reindex(range(24), fill_value=0)
    .rename("suspicious_events")
)

suspicious_by_hour

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))

suspicious_by_hour.plot(kind="bar", ax=ax, color="#d62728")
ax.set_title("Suspicious activity by hour")
ax.set_xlabel("Hour of day")
ax.set_ylabel("Event count")
ax.set_xticklabels([f"{hour:02d}:00" for hour in suspicious_by_hour.index], rotation=45, ha="right")
ax.grid(axis="y", alpha=0.3)

if suspicious_by_hour.sum() == 0:
    ax.text(0.5, 0.5, "No suspicious events found", transform=ax.transAxes, ha="center", va="center")

plt.tight_layout()
plt.show()